In [1]:
# import libraries
from sklearn.metrics import f1_score
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import warnings
warnings.filterwarnings("ignore")

# set config
results_df = pd.DataFrame(columns=["prompt_template", "prompt_name", "F1_score"])
model_name = "google/flan-t5-small"


/Users/negin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/negin/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
valid_df = pd.read_csv('valid_df_with_predictions.csv')
valid_df = valid_df[['sentiment', 'text','out_label_LR']]

In [12]:
valid_df

,sentiment,text,out_label_LR
0,positive,These work great and are a better value than m...,positive
1,negative,I replaced the all the suspension rods/shocks ...,positive
2,positive,my old ones couldn't tolerate the heat and bec...,positive
3,negative,This bracket will not fit the under counter wa...,negative
4,positive,These work fairly well and save a lot of money...,negative
...,...,...,...
695,negative,Installed this in my frig and a gush of black ...,negative
696,negative,Does not fit my Samsung refrigerator.,negative
697,positive,The video showing how it works was extremely a...,positive
698,positive,"Did have a small issue putting it in, but it w...",positive


In [47]:
# Load the dataset
valid_df = pd.read_csv('data/6/valid.csv')
valid_df.drop(columns=['data_id'], inplace=True)
valid_df.dropna(inplace=True)

KeyError: "['data_id'] not found in axis"

In [4]:
# data cleaning
def clean_text(df, text_col="text", target_col="sentiment"):

    df = df.dropna(subset=[text_col])
    df[text_col] = df[text_col].str.strip()
    df[text_col] = df[text_col].str[:2048]

    df[target_col] = df[target_col].str.lower()
    return df

def post_processing(x):
    x = x.lower()
    if 'positive' not in x and 'negative' not in x:
        return 'positive'
    pos_idx = x.find('positive')
    neg_idx = x.find('negative')
    if neg_idx != -1 and (pos_idx == -1 or neg_idx < pos_idx):
        return 'negative'
    return 'positive'

# inference
def batch_predict(texts, prompt_template, tokenizer, model):
    prompts = [prompt_template.format(t=t) for t in texts]

    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False
        )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

def pipeline(df, prompt_template, prompt_name):
    # data cleaning
    df = clean_text(df)

    # Load model
    model_name = "google/flan-t5-small"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    # Set model to eval mode (faster)
    model.eval()

    # Apply in batches
    batch_size = 32
    preds = []

    for i in range(0, len(df), batch_size):
        batch_texts = df["text"].iloc[i:i+batch_size].tolist()
        preds.extend(batch_predict(batch_texts, prompt_template, tokenizer, model))

    df[prompt_name] = [p.strip() for p in preds]

    # post-processing
    df[prompt_name] = df[prompt_name].apply(post_processing)
    
    # evaluation
    df[prompt_name] = df[prompt_name].str.lower()
    df['sentiment'] = df['sentiment'].str.lower()
    df['correct'] = df['sentiment'] == df[prompt_name]

    f1 = f1_score(df['sentiment'], df[prompt_name], average='weighted')
    print(f"F1-Score: {round(100*f1, 1)}%")

    results = {
        "prompt_template": prompt_template,
        "prompt_name": prompt_name,
        "F1_score": f1
    }

    return results, df

In [5]:
# one shot
# f1 = 90.7%
prompt = """You are a sentiment classifier.
Answer with only one word: positive or negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: positive

Text: {t}
Answer:"""

prompt_name = "out_label_Prompt_1"
results, df = pipeline(valid_df, prompt, prompt_name)
results_df = pd.concat([results_df, pd.DataFrame([results])], ignore_index=True)
valid_df = valid_df.merge(df[prompt_name], left_index=True, right_index=True, how="left")

F1-Score: 91.0%


In [6]:
# however
# f1 = 91.4%
prompt = """Classify the sentiment of the text.

Labels: positive , negative

Answer with exactly one word: positive or negative. Do not explain.
After "but", "however", or "although", the following sentiment is more important.

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Answer: positive

Text: {t}
Answer:"""

prompt_name = "out_label_Prompt_2"
results, df = pipeline(valid_df, prompt, prompt_name)
results_df = pd.concat([results_df, pd.DataFrame([results])], ignore_index=True)
valid_df = valid_df.merge(df[prompt_name], left_index=True, right_index=True, how="left")

F1-Score: 91.4%


In [7]:
# two shot
# f1 = 89.5%
prompt = """You are a sentiment classifier.
Answer with only one word: positive or negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: positive

Text: Bought it June 21, died last night. No longevity. Worked good other then that.
Answer: negative

Text: {t}
Answer:"""

prompt_name = "out_label_Prompt_3"
results, df = pipeline(valid_df, prompt, prompt_name)
results_df = pd.concat([results_df, pd.DataFrame([results])], ignore_index=True)
valid_df = valid_df.merge(df[prompt_name], left_index=True, right_index=True, how="left")

F1-Score: 89.8%


In [8]:
# question
# f1 = 89.0%
prompt = """
What is the sentiment of this text?

Answer using only one word: positive or negative.

Text: {t}
Answer:"""

prompt_name = "out_label_Prompt_4"
results, df = pipeline(valid_df, prompt, prompt_name)
results_df = pd.concat([results_df, pd.DataFrame([results])], ignore_index=True)
valid_df = valid_df.merge(df[prompt_name], left_index=True, right_index=True, how="left")

F1-Score: 89.0%


In [9]:
# explain sentiment
# f1 = 88.7%
prompt = """
Classify sentiment.

Positive = praise, satisfaction, good experience.
Negative = complaint, disappointment, bad experience.

Answer with one word only: positive or negative.

Text: {t}
Answer:"""
prompt_name = "out_label_Prompt_5"
results, df = pipeline(valid_df, prompt, prompt_name)
results_df = pd.concat([results_df, pd.DataFrame([results])], ignore_index=True)
valid_df = valid_df.merge(df[prompt_name], left_index=True, right_index=True, how="left")

F1-Score: 88.7%


In [10]:
valid_df

,sentiment,text,out_label_LR,out_label_Prompt_1,out_label_Prompt_2,out_label_Prompt_3,out_label_Prompt_4,out_label_Prompt_5
0,positive,These work great and are a better value than m...,positive,positive,positive,positive,positive,positive
1,negative,I replaced the all the suspension rods/shocks ...,positive,negative,negative,negative,negative,negative
2,positive,my old ones couldn't tolerate the heat and bec...,positive,positive,positive,positive,positive,positive
3,negative,This bracket will not fit the under counter wa...,negative,negative,negative,negative,negative,negative
4,positive,These work fairly well and save a lot of money...,negative,positive,positive,positive,positive,positive
...,...,...,...,...,...,...,...,...
695,negative,Installed this in my frig and a gush of black ...,negative,negative,negative,negative,negative,negative
696,negative,Does not fit my Samsung refrigerator.,negative,negative,negative,negative,negative,negative
697,positive,The video showing how it works was extremely a...,positive,positive,positive,positive,positive,positive
698,positive,"Did have a small issue putting it in, but it w...",positive,positive,positive,positive,positive,positive


In [14]:
results_df.to_csv('results_gpt.csv', index=False)

In [13]:
valid_df.to_csv("valid_df_with_predictions.csv", index=False)

In [44]:
valid_df.columns

Index(['sentiment', 'text', 'out_label_Prompt_5', 'out_label_Prompt_4',
       'out_label_Prompt_3', 'out_label_Prompt_2', 'out_label_Prompt_1'],
      dtype='object')

## Final code

In [3]:
prompt_templates = {
    "out_label_Prompt_1": """You are a sentiment classifier.
    Answer with only one word: positive or negative.

    Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
    Answer: positive

    Text: {t}
    Answer:""",
    "out_label_Prompt_2": """Classify the sentiment of the text.

    Labels: positive , negative

    Answer with exactly one word: positive or negative. Do not explain.
    After "but", "however", or "although", the following sentiment is more important.

    Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
    Answer: positive

    Text: {t}
    Answer:""",
    "out_label_Prompt_3": """You are a sentiment classifier.
    Answer with only one word: positive or negative.

    Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
    Answer: positive

    Text: Bought it June 21, died last night. No longevity. Worked good other then that.
    Answer: negative

    Text: {t}
    Answer:""",
    "out_label_Prompt_4":  """What is the sentiment of this text?

    Answer using only one word: positive or negative.

    Text: {t}
    Answer:""",
    "out_label_Prompt_5":  """Classify sentiment.

    Positive = praise, satisfaction, good experience.
    Negative = complaint, disappointment, bad experience.

    Answer with one word only: positive or negative.

    Text: {t}
    Answer:"""
}

import time
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import f1_score

# data cleaning
def clean_text(df, text_col="text", target_col="sentiment"):

    df = df.dropna(subset=[text_col])
    df[text_col] = df[text_col].str.strip()
    df[text_col] = df[text_col].str[:2048]

    df[target_col] = df[target_col].str.lower()
    return df

def post_processing(x):
    x = x.lower()
    if 'positive' not in x and 'negative' not in x:
        return 'positive'
    pos_idx = x.find('positive')
    neg_idx = x.find('negative')
    if neg_idx != -1 and (pos_idx == -1 or neg_idx < pos_idx):
        return 'negative'
    return 'positive'


# inference
def batch_predict(texts, prompt_template, tokenizer, model):
    prompts = [prompt_template.format(t=t) for t in texts]

    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False
        )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)


def train_unsup(train_file, val_file, model_dir):

    # Read training and validation set data
    df = pd.read_csv(val_file)
    print('Validation set has', len(df),'data points')

    # load model from local or download from HuggingFace
    model_name = "google/flan-t5-small"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    # data cleaning
    df = clean_text(df)

    # Set model to eval mode
    model.eval()

    for prompt_name, prompt in prompt_templates.items():
        # Apply in batches
        batch_size = 32
        preds = []

        for i in range(0, len(df), batch_size):
            batch_texts = df["text"].iloc[i:i+batch_size].tolist()
            preds.extend(batch_predict(batch_texts, prompt, tokenizer, model))

        df[prompt_name] = [p.strip() for p in preds]
        
        # post-processing
        df[prompt_name] = df[prompt_name].apply(post_processing)

        # evaluation
        # if "sentiment" in test_df.columns:
        df[prompt_name] = df[prompt_name].str.lower()
        df['sentiment'] = df['sentiment'].str.lower()

        f1 = f1_score(df['sentiment'], df[prompt_name], average='weighted')
        print(f"{prompt_name} has F1-Score of: {round(100*f1, 1)}%")

val_file = 'data/inputs/valid.csv'
train_unsup(None, val_file, None)


Validation set has 700 data points
out_label_Prompt_1 has F1-Score of: 91.0%
out_label_Prompt_2 has F1-Score of: 91.4%
out_label_Prompt_3 has F1-Score of: 89.8%
out_label_Prompt_4 has F1-Score of: 89.0%
out_label_Prompt_5 has F1-Score of: 88.7%
